In [1]:
%pip install -U pandas
import pandas as pd
print(pd.__version__)


2.3.3


## What does this code do

- Computes hourly plant power for CRAH + water-side economizer (WSE) + chiller + tower + pumps.

- Uses ε–NTU to see how much the WSE can precool; the chiller picks up the rest; classifies Mode 1/2/3 accordingly.

- Scales fans/pumps with affinity-law (≈ cubic) from their rated values.

In [3]:
from dataclasses import dataclass, field
import numpy as np
import pandas as pd

@dataclass
class CRACCoeffs:
    """Eq. (3) coefficients for EER(Twb_in, Tdb_out) from the paper."""
    a1: float = 1.35215
    a2: float = 0.10195
    a3: float = -0.00025
    a4: float = -0.06681
    a5: float = 0.00085
    a6: float = -0.00136

@dataclass
class CRACParams:
    Qrate_kW: float  # Rated cooling capacity (kW) for the CRAC plant
    coeffs: CRACCoeffs = field(default_factory=CRACCoeffs)  
    aux_fraction: float = 0.10   # UPS/PDU + lighting ≈ 10% of servers (Eq. 2)

def compute_Qsum(server_kW: np.ndarray, aux_fraction: float = 0.10) -> np.ndarray:
    """
    Eq. (2): Q_sum = Q_server + Q_UPS + Q_PDU + Q_light.
    Paper assumes UPS/PDU + lighting ≈ 10% of server load.
    Returns kW array of total cooling load.
    """
    server_kW = np.asarray(server_kW, dtype=float)
    return server_kW * (1.0 + aux_fraction)

def eer_crac(Twb_in_C: np.ndarray, Tdb_out_C: np.ndarray, c: CRACCoeffs) -> np.ndarray:
    """
    Eq. (3): EER = a1 + a2*Twb,in + a3*Twb,in^2 + a4*Tdb,out + a5*Tdb,out^2 + a6*Twb,in*Tdb,out.
    Twb_in_C: indoor wet-bulb (°C); Tdb_out_C: outdoor dry-bulb (°C) as per paper’s notation.
    Returns EER (dimensionless).
    """
    Twb = np.asarray(Twb_in_C, dtype=float)
    Tdb = np.asarray(Tdb_out_C, dtype=float)
    return (c.a1 + c.a2*Twb + c.a3*Twb**2 + c.a4*Tdb + c.a5*Tdb**2 + c.a6*Twb*Tdb)

def plr(Qsum_kW: np.ndarray, Qrate_kW: float) -> np.ndarray:
    """
    Eq. (6): PLR = Q_sum / Q_rate. Clamped to (0, 1] for numerical stability.
    """
    Qsum = np.asarray(Qsum_kW, dtype=float)
    plr_vals = Qsum / float(Qrate_kW)
    return np.clip(plr_vals, 1e-6, 1.0)

def plf(plr_vals: np.ndarray) -> np.ndarray:
    """
    Eq. (7): PLF = 0.7824*exp(0.2818*PLR) - 0.7575*exp(-69.7*PLR).
    """
    plr_vals = np.asarray(plr_vals, dtype=float)
    return 0.7824*np.exp(0.2818*plr_vals) - 0.7575*np.exp(-69.7*plr_vals)

def p_crac_kW(Qsum_kW: np.ndarray, EER: np.ndarray, PLF: np.ndarray) -> np.ndarray:
    """
    Eq. (8): P_CRAC (kW) = Q_sum / (EER * PLF).
    """
    Qsum_kW = np.asarray(Qsum_kW, dtype=float)
    denom = np.asarray(EER, dtype=float) * np.asarray(PLF, dtype=float)
    # avoid divide-by-zero
    denom = np.where(denom <= 1e-9, 1e-9, denom)
    return Qsum_kW / denom

def crac_power_timeseries(df: pd.DataFrame, params: CRACParams) -> pd.DataFrame:
    """
    Compute CRAC power per timestep using §3.3 pipeline.
    Required columns in df:
      - 'Qserver_kW' (server IT load)
      - 'Twb_in_C'   (indoor wet-bulb)
      - 'Tdb_out_C'  (outdoor dry-bulb)
    Returns df with added columns: Qsum_kW, PLR, PLF, EER, P_CRAC_kW.
    """
    out = df.copy()
    out['Qsum_kW'] = compute_Qsum(out['Qserver_kW'].values, aux_fraction=params.aux_fraction)
    out['EER'] = eer_crac(out['Twb_in_C'].values, out['Tdb_out_C'].values, params.coeffs)
    out['PLR'] = plr(out['Qsum_kW'].values, params.Qrate_kW)
    out['PLF'] = plf(out['PLR'].values)
    out['P_CRAC_kW'] = p_crac_kW(out['Qsum_kW'].values, out['EER'].values, out['PLF'].values)
    return out

# Example test
import pandas as pd

# small fake dataset
hourly = pd.DataFrame({
    'Qserver_kW': [1200, 1250, 1300, 1400],
    'Twb_in_C':   [14.0, 14.2, 14.0, 15.0],
    'Tdb_out_C':  [32.0, 33.5, 31.0, 30.0],
})

params = CRACParams(Qrate_kW=1800.0)

result = crac_power_timeseries(hourly, params)

# show calculated columns
print(result[['Qsum_kW', 'EER', 'PLR', 'PLF', 'P_CRAC_kW']])



   Qsum_kW       EER       PLR       PLF    P_CRAC_kW
0   1320.0  0.853650  0.733333  0.962005  1607.373642
1   1375.0  0.818256  0.763889  0.970324  1731.797084
2   1430.0  0.885950  0.794444  0.978715  1649.189448
3   1540.0  0.973850  0.855556  0.995716  1588.156663


In [4]:
# --- debug: print one row's EER exactly as Eq.(3)
row0_eer = eer_crac([result.loc[0,'Twb_in_C']],
                    [result.loc[0,'Tdb_out_C']],
                    params.coeffs)[0]
print("Row0 EER =", row0_eer)  # expect ~0.85365 with Twb_in=14, Tdb_out=32

# --- optional: allow using OUTDOOR WET-BULB instead of dry-bulb (paper text is ambiguous)
def crac_power_timeseries(df, params, treat_outdoor_as='drybulb', eer_scale=1.0):
    out = df.copy()
    out['Qsum_kW'] = compute_Qsum(out['Qserver_kW'].values, aux_fraction=params.aux_fraction)

    # choose outdoor temperature input for Eq.(3)
    if treat_outdoor_as == 'wetbulb' and 'Twb_out_C' in out.columns:
        T_out = out['Twb_out_C'].values
    else:
        T_out = out['Tdb_out_C'].values

    EER = eer_crac(out['Twb_in_C'].values, T_out, params.coeffs) * float(eer_scale)
    out['EER'] = EER
    out['PLR'] = plr(out['Qsum_kW'].values, params.Qrate_kW)
    out['PLF'] = plf(out['PLR'].values)
    out['P_CRAC_kW'] = p_crac_kW(out['Qsum_kW'].values, out['EER'].values, out['PLF'].values)
    return out

# --- optional: one-point calibration to match a known EER (e.g., 1.85 at a specific condition)
def calibrate_eer_scale(target_eer, Twb_in, T_out, coeffs):
    base = eer_crac([Twb_in],[T_out], coeffs)[0]
    return target_eer / base if base > 0 else 1.0

# Example: force EER=1.85 at Twb_in=14, outdoor=27.8 (if you want this narrative)
scale = calibrate_eer_scale(1.85, 14.0, 27.8, params.coeffs)
print("EER scale =", scale)

# Recompute with chosen interpretation/scale
result2 = crac_power_timeseries(hourly, params,
                                treat_outdoor_as='wetbulb',  # or 'wetbulb' if you add Twb_out_C
                                eer_scale=scale)             # or 1.0 to keep the paper exactly
print(result2[['Qsum_kW','EER','PLR','PLF','P_CRAC_kW']])


Row0 EER = 0.85365
EER scale = 1.84864309596756
   Qsum_kW       EER       PLR       PLF   P_CRAC_kW
0   1320.0  1.578094  0.733333  0.962005  869.488354
1   1375.0  1.512662  0.763889  0.970324  936.793634
2   1430.0  1.637805  0.794444  0.978715  892.108083
3   1540.0  1.800301  0.855556  0.995716  859.093173


In [5]:
# ---------- Physical constants ----------
WATER_CP_KJ_PER_KG_K = 4.186
WATER_DENS_KG_PER_M3 = 1000.0

# ---------- Chiller COP surfaces (paper Eq. 4 & 5) ----------
def cop_centrifugal(Tctw_C, PLR):
    # COPv = b1 + b2*Tctw + b3*PLR + b4*Tctw^2 + b5*Tctw*PLR + b6*PLR^2
    b1, b2, b3, b4, b5, b6 = 25.47, -1.066, 6.335, 0.01188, 0.1263, -7.581
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return b1 + b2*T + b3*L + b4*T*T + b5*T*L + b6*L*L

def cop_magnetic(Tctw_C, PLR):
    # COPm = c1 + c2*Tctw + c3*PLR + c4*Tctw^2 + c5*Tctw*PLR + c6*PLR^2
    c1, c2, c3, c4, c5, c6 = 25.17, -0.7536, -1.081, 0.005736, 0.1851, -4.568
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return c1 + c2*T + c3*L + c4*T*T + c5*T*L + c6*L*L

In [6]:
def hx_eps_ntu_Q(C_hot, C_cold, UA_kW_per_K, Th_in, Tc_in):
    """
    Vectorized ε–NTU heat exchanger.
    Inputs can be arrays or scalars. Units: C_* in kW/K, UA in kW/K, temps °C.
    Returns arrays: (Q_kW, Th_out, Tc_out).
    """
    C_hot = np.asarray(C_hot, dtype=float)
    C_cold = np.asarray(C_cold, dtype=float)
    Th_in = np.asarray(Th_in, dtype=float)
    Tc_in = np.asarray(Tc_in, dtype=float)
    UA = float(UA_kW_per_K)

    eps = 1e-9
    C_min = np.minimum(C_hot, C_cold)
    C_max = np.maximum(C_hot, C_cold)
    Cr = np.divide(C_min, C_max, out=np.zeros_like(C_min), where=(C_max > eps))
    NTU = UA / np.maximum(C_min, eps)

    term = np.exp(-NTU * (1.0 - Cr))
    denom = 1.0 - Cr * term
    eff = np.divide(1.0 - term, denom, out=np.zeros_like(denom), where=(denom > eps))

    Qmax = C_min * np.maximum(0.0, Th_in - Tc_in)
    Q = eff * Qmax
    Th_out = np.where(C_hot > eps, Th_in - Q / np.maximum(C_hot, eps), Th_in)
    Tc_out = np.where(C_cold > eps, Tc_in + Q / np.maximum(C_cold, eps), Tc_in)
    return Q, Th_out, Tc_out

In [7]:
# ---------- Utility ----------
def k_cubic(x):
    """Vectorized cubic scaling (0..1) for arrays or scalars."""
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 0.0, 1.0)
    return x**3

In [8]:
@dataclass
class PlantParams:
    # Setpoints
    Tcw_supply_set_C: float = 7.0        # chilled-water supply setpoint
    dT_chilled_K: float = 5.0            # nominal ΔT on chilled-water loop
    dT_cooling_K: float = 5.0            # nominal ΔT on cooling-water loop
    tower_approach_K: float = 4.0        # T_ctw_in ≈ Twb_out + approach
    tower_Tmin_C: float = 2.0            # anti-freeze lower bound at tower outlet
    UA_wse_kW_per_K: float = 4000.0/1.0  # from Table 4: UA=4000 kW/ΔT → write as kW/K

    # CRAH bank (capacity & fan power)
    crah_caps_kW: tuple = (72*102.0, 29*62.0, 12*20.5)  # capacities
    crah_fans_kW: tuple = (72*5.66, 29*3.44, 12*1.40)    # rated fan power
    # Pumps (rated)
    chw_flow_m3_per_h_per_pump: float = 660.0
    chw_pumps_use: int = 2
    chw_pump_power_kW_per_pump: float = 75.0

    ctw_flow_m3_per_h_per_pump: float = 900.0
    ctw_pumps_use: int = 2
    ctw_pump_power_kW_per_pump: float = 55.0

    # Tower fans (rated)
    tower_fan_power_kW_per_ct: float = 37.5
    tower_cells_use: int = 2

    # Chillers
    chiller_type: str = "centrifugal"    # "centrifugal" or "magnetic"
    chiller_units_use: int = 2
    chiller_cap_kW_per_unit: float = 4058.0    # Plan 1; for magnetic set to 3900.0
    # Optional: cap for COP to avoid nonsense at weird PLR/T
    COP_min: float = 2.0
    COP_max: float = 18.0

In [9]:
def plan_power_timeseries(df, params: PlantParams):
    """
    Inputs required columns per row:
      - Qserver_kW : IT load (kW)
      - Twb_out_C  : outdoor wet-bulb (°C)  (for tower sink)
      - (optional) Tdb_out_C, Twb_in_C if you also run the CRAC block separately

    Returns a DataFrame with per-step mode, Q splits, and power components.
    """
    out = df.copy()
    # Total cooling load ≈ servers + 10% aux (UPS/PDU+lighting) per paper Eq.(2)
    Qsum = out['Qserver_kW'].to_numpy(float) * 1.10

    # CRAH fan power scaling
    crah_Q_cap = sum(params.crah_caps_kW)
    crah_fan_rated = sum(params.crah_fans_kW)
    kfan_crah = np.clip(Qsum / max(1.0, crah_Q_cap), 0, 1)
    P_crah = k_cubic(kfan_crah) * crah_fan_rated  # simple affinity law

    # Determine chilled-water return based on set ΔT & required flow
    Cpw = WATER_CP_KJ_PER_KG_K / 1.0  # kJ/(kg*K); we'll convert to kW/K via kg/s
    # Required chilled-water mass flow to meet Qsum with ΔT_chilled
    m_chw_req = (Qsum) / (Cpw * params.dT_chilled_K)  # kg/s because Q(kW) / (kJ/kgK * K) = kg/s
    # Rated available mass flow from pumps in use
    chw_flow_total_m3h = params.chw_flow_m3_per_h_per_pump * params.chw_pumps_use
    m_chw_rated = chw_flow_total_m3h * WATER_DENS_KG_PER_M3 / 3600.0  # kg/s
    kflow_chw = np.clip(m_chw_req / max(1e-9, m_chw_rated), 0, 1)
    # Effective hot-side capacity rate (kW/K)
    C_hot = np.minimum(m_chw_req, m_chw_rated) * Cpw  # kJ/sK == kW/K
    Tcw_r = params.Tcw_supply_set_C + params.dT_chilled_K

    # Cooling tower sink temperature (tower outlet to condenser/WSE)
    Tctw_in = np.maximum(out['Twb_out_C'].to_numpy(float) + params.tower_approach_K, params.tower_Tmin_C)
    # Cooling-water side available capacity (use rated flow as baseline)
    ctw_flow_total_m3h = params.ctw_flow_m3_per_h_per_pump * params.ctw_pumps_use
    m_ctw_rated = ctw_flow_total_m3h * WATER_DENS_KG_PER_M3 / 3600.0
    C_cold_rated = m_ctw_rated * Cpw  # kW/K

    # Heat exchanger (WSE) potential transfer
    Q_wse, Th_out, Tc_out = hx_eps_ntu_Q(
        C_hot=C_hot,
        C_cold=C_cold_rated,
        UA_kW_per_K=params.UA_wse_kW_per_K,
        Th_in=Tcw_r,
        Tc_in=Tctw_in
    )

    # Clamp WSE to not overcool below supply setpoint
    Q_needed_to_set = np.maximum(0.0, (Tcw_r - params.Tcw_supply_set_C) * C_hot)
    Q_wse = np.minimum(Q_wse, Q_needed_to_set)
    Q_wse = np.minimum(Q_wse, Qsum)  # cannot exceed load

    # Determine mode + chiller portion
    T_hot_out = Tcw_r - Q_wse / np.maximum(C_hot, 1e-9)
    mode = np.where(Q_wse >= Qsum - 1e-6, "MODE1_FULL_FREE",
             np.where(Q_wse > 1e-6, "MODE2_PARTIAL_FREE", "MODE3_CHILLER"))

    Q_ch = np.maximum(0.0, Qsum - Q_wse)  # kW to chiller

    # Chiller COP & power (runs in Mode 2 & 3)
    # Select cooling-water temperature to chiller condenser: use tower outlet Tctw_in
    # Active units & PLR
    cap_unit = params.chiller_cap_kW_per_unit
    units = params.chiller_units_use
    total_cap = cap_unit * units
    PLR_total = np.clip(Q_ch / max(1e-9, total_cap), 1e-6, 1.0)

    if params.chiller_type.lower().startswith('mag'):
        COP = cop_magnetic(Tctw_in, PLR_total)
    else:
        COP = cop_centrifugal(Tctw_in, PLR_total)
    COP = np.clip(COP, params.COP_min, params.COP_max)

    P_ch = np.where(Q_ch > 0.0, Q_ch / COP, 0.0)  # kW

    # Cooling-tower heat rejection estimate (for fan scaling): Q_rej ≈ Q_ch*(COP-1)/COP + Q_wse
    Q_rej = Q_wse + np.where(Q_ch > 0.0, Q_ch * (COP - 1.0) / COP, 0.0)  # kW to cooling-water loop
    # Rated tower thermal capacity (from rated flow & ΔT)
    Q_tower_rated = C_cold_rated * params.dT_cooling_K  # kW
    ktower = np.clip(Q_rej / max(Q_tower_rated, 1e-9), 0, 1)
    P_tower_fans = k_cubic(ktower) * (params.tower_fan_power_kW_per_ct * params.tower_cells_use)

    # Pump powers (affinity law scaling)
    P_chw_pumps = k_cubic(kflow_chw) * (params.chw_pump_power_kW_per_pump * params.chw_pumps_use)
    # Approximate cooling-water flow fraction from Q_rej
    kflow_ctw = np.clip(Q_rej / max(Q_tower_rated, 1e-9), 0, 1)
    P_ctw_pumps = k_cubic(kflow_ctw) * (params.ctw_pump_power_kW_per_pump * params.ctw_pumps_use)

    # Sum plant power
    P_sys = P_ch + P_chw_pumps + P_ctw_pumps + P_tower_fans + P_crah

    # Pack outputs
    out = out.assign(
        Qsum_kW=Qsum,
        mode=mode,
        Q_wse_kW=Q_wse,
        Q_chiller_kW=Q_ch,
        COP=np.where(Q_ch > 0.0, COP, np.nan),
        P_chiller_kW=P_ch,
        P_crah_kW=P_crah,
        P_pumps_chw_kW=P_chw_pumps,
        P_pumps_ctw_kW=P_ctw_pumps,
        P_tower_fans_kW=P_tower_fans,
        P_sys_kW=P_sys,
        T_ctw_in_C=Tctw_in,
        T_cw_return_C=Tcw_r,
        T_cw_after_WSE_C=T_hot_out
    )
    return out


In [10]:
# ------------------ Quick demo (remove/adjust as needed) ------------------

hourly2 = pd.DataFrame({
     'Qserver_kW': [4000, 4200, 4500, 3800],  # IT load
     'Twb_out_C':  [18.0, 20.0, 25.0, 12.0],  # outdoor wet-bulb for tower
 })

p = PlantParams(
     chiller_type="magnetic",           # "centrifugal" for Plan 1
     chiller_cap_kW_per_unit=3900.0,    # 4058.0 for centrifugal (Plan 1), 3900.0 for magnetic (Plan 2)
 )
 
res = plan_power_timeseries(hourly2, p)

print(res[['mode','Qsum_kW','Q_wse_kW','Q_chiller_kW','COP','P_sys_kW']])

            mode  Qsum_kW  Q_wse_kW  Q_chiller_kW        COP    P_sys_kW
0  MODE3_CHILLER   4400.0       0.0        4400.0  11.600777  472.002770
1  MODE3_CHILLER   4620.0       0.0        4620.0  10.775935  535.804351
2  MODE3_CHILLER   4950.0       0.0        4950.0   9.020408  679.592895
3  MODE3_CHILLER   4180.0       0.0        4180.0  14.276759  372.763134


In [11]:
# stakeholder View -------

res['Qserver_kW'] = res['Qsum_kW'] / 1.10
res['P_chiller_kW_est'] = np.where(res['Q_chiller_kW']>0,
                                   res['Q_chiller_kW']/res['COP'], 0.0)
res['P_aux_kW'] = res['P_sys_kW'] - res['P_chiller_kW_est']
res['Cooling_overhead_%'] = 100.0 * res['P_sys_kW'] / res['Qserver_kW']
res['PUE_prime'] = 1.0 + res['P_sys_kW'] / res['Qserver_kW']
print(res[['mode','Qserver_kW','Qsum_kW','Q_wse_kW','Q_chiller_kW',
           'COP','P_chiller_kW_est','P_aux_kW','P_sys_kW',
           'Cooling_overhead_%','PUE_prime']])


            mode  Qserver_kW  Qsum_kW  Q_wse_kW  Q_chiller_kW        COP  \
0  MODE3_CHILLER      4000.0   4400.0       0.0        4400.0  11.600777   
1  MODE3_CHILLER      4200.0   4620.0       0.0        4620.0  10.775935   
2  MODE3_CHILLER      4500.0   4950.0       0.0        4950.0   9.020408   
3  MODE3_CHILLER      3800.0   4180.0       0.0        4180.0  14.276759   

   P_chiller_kW_est    P_aux_kW    P_sys_kW  Cooling_overhead_%  PUE_prime  
0        379.284955   92.717815  472.002770           11.800069   1.118001  
1        428.733101  107.071250  535.804351           12.757246   1.127572  
2        548.755696  130.837199  679.592895           15.102064   1.151021  
3        292.783542   79.979592  372.763134            9.809556   1.098096  


In [12]:
# Demo inputs forcing: Mode1 (full free), Mode2 (partial), then two Mode3 (chiller)

demo = pd.DataFrame({
    'Qserver_kW': [4000, 4000, 4000, 4000],
    'Twb_out_C':  [0.0,   6.0,   18.0,  12.0],  # cold → mild → hot
})

# Plan 2 (magnetic) and Plan 1 (centrifugal)
p_mag = PlantParams(chiller_type="magnetic",    chiller_cap_kW_per_unit=3900.0)
p_cen = PlantParams(chiller_type="centrifugal", chiller_cap_kW_per_unit=4058.0)

# Run both plans
res_mag = plan_power_timeseries(demo, p_mag)
res_cen = plan_power_timeseries(demo, p_cen)

print("MAGNETIC:")
print(res_mag[['mode','Qsum_kW','Q_wse_kW','Q_chiller_kW','COP','P_sys_kW']], "\n")
print("CENTRIFUGAL:")
print(res_cen[['mode','Qsum_kW','Q_wse_kW','Q_chiller_kW','COP','P_sys_kW']], "\n")

# Side-by-side comparison and savings
view = pd.DataFrame({
    'mode': res_mag['mode'],
    'IT_kW': res_mag['Qsum_kW'] / 1.10,
    'Twb_out_C': demo['Twb_out_C'],
    'P_sys_mag_kW': res_mag['P_sys_kW'],
    'P_sys_cen_kW': res_cen['P_sys_kW'],
})
view['Δ_kW (cen - mag)'] = view['P_sys_cen_kW'] - view['P_sys_mag_kW']
print("SIDE-BY-SIDE:")
print(view, "\n")

# Totals (assume 1 hour per row)
dt_hours = 1.0
kWh_mag = float((res_mag['P_sys_kW'] * dt_hours).sum())
kWh_cen = float((res_cen['P_sys_kW'] * dt_hours).sum())
delta_kWh = kWh_cen - kWh_mag

elec_price = 0.10  # $/kWh — change to your tariff
delta_usd = delta_kWh * elec_price
pct_savings = 100.0 * delta_kWh / kWh_cen if kWh_cen > 0 else float('nan')

print("=== TOTALS ===")
print(f"Magnetic   total kWh: {kWh_mag:,.1f}")
print(f"Centrifugal total kWh: {kWh_cen:,.1f}")
print(f"ΔkWh (cen - mag):     {delta_kWh:,.1f}")
print(f"Δ$ at ${elec_price}/kWh: ${delta_usd:,.2f}  ({pct_savings:.2f}% savings)")


MAGNETIC:
                 mode  Qsum_kW     Q_wse_kW  Q_chiller_kW        COP  \
0     MODE1_FULL_FREE   4400.0  4400.000000      0.000000        NaN   
1  MODE2_PARTIAL_FREE   4400.0  1684.518747   2715.481253  17.922023   
2       MODE3_CHILLER   4400.0     0.000000   4400.000000  11.600777   
3       MODE3_CHILLER   4400.0     0.000000   4400.000000  14.188076   

     P_sys_kW  
0   95.975975  
1  246.120281  
2  472.002770  
3  403.388211   

CENTRIFUGAL:
                 mode  Qsum_kW     Q_wse_kW  Q_chiller_kW        COP  \
0     MODE1_FULL_FREE   4400.0  4400.000000      0.000000        NaN   
1  MODE2_PARTIAL_FREE   4400.0  1684.518747   2715.481253  17.691502   
2       MODE3_CHILLER   4400.0     0.000000   4400.000000  10.480591   
3       MODE3_CHILLER   4400.0     0.000000   4400.000000  13.757118   

     P_sys_kW  
0   95.975975  
1  248.077293  
2  512.227310  
3  413.024579   

SIDE-BY-SIDE:
                 mode   IT_kW  Twb_out_C  P_sys_mag_kW  P_sys_cen_kW  \
0    

In [13]:
# =============================================================================
# BLOCK 1 — INPUTS (paths, tariffs, finance knobs)
# Purpose: define data locations and scenario knobs in one place.
# =============================================================================
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 6)

# 1A) Paths to 8760 Excel files (you gave these)
PHOENIX_XLSX   = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx"
FAIRBANKS_XLSX = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx"

# 1B) Site tariffs (flat) — used only in annual cost calc
SITES = {
    "Phoenix":   {"energy_price_usd_per_kWh": 0.10},
    "Fairbanks": {"energy_price_usd_per_kWh": 0.08},
}

# 1C) Finance knobs (for later NPV/IRR blocks if needed)
DISCOUNT_RATE = 0.05     # 5% real
ESCALATION    = 0.02     # 2% real
LIFE_YEARS    = 20

# =============================================================================
# BLOCK 2 — DATA LOADER
# Purpose: read your Excel files and standardize columns for the engine.
# Expect input cols: ["Date/Time","Qserver_kW","Twb_in_C","Tdb_out_C"].
# For dispatch we need outdoor wet-bulb; until we have RH we use Tdb as proxy:
#   Twb_out_C ≈ Tdb_out_C   (replace later when you add RH).
# =============================================================================
def load_site_frame(xlsx_path: str) -> pd.DataFrame:
    df = pd.read_excel(xlsx_path)

    # Normalize column names
    df.columns = [c.strip().replace(" ", "") for c in df.columns]

    # Best-effort datetime (not strictly required)
    if "Date/Time" in df.columns:
        df["DateTime"] = pd.to_datetime(df["Date/Time"], errors="coerce", format="mixed")
    else:
        df["DateTime"] = pd.NaT

    out = pd.DataFrame({
        "DateTime":   df["DateTime"],
        "Qserver_kW": pd.to_numeric(df["Qserver_kW"], errors="coerce"),
        "Tdb_out_C":  pd.to_numeric(df["Tdb_out_C"],  errors="coerce"),
        "Twb_in_C":   pd.to_numeric(df.get("Twb_in_C", np.nan), errors="coerce"),
    })
    # Approximate outdoor wet-bulb with dry-bulb (temporary)
    out["Twb_out_C"] = out["Tdb_out_C"]

    # Keep exactly 8760 rows (truncate or error if fewer)
    out = out.dropna(subset=["Qserver_kW", "Tdb_out_C"]).reset_index(drop=True)
    if len(out) < 8760:
        raise ValueError(f"{xlsx_path} has only {len(out)} usable rows; need 8760.")
    if len(out) > 8760:
        out = out.iloc[:8760].copy()

    return out

df_phoenix   = load_site_frame(PHOENIX_XLSX)
df_fairbanks = load_site_frame(FAIRBANKS_XLSX)
print("Loaded source — Phoenix:", PHOENIX_XLSX)
print("Loaded source — Fairbanks:", FAIRBANKS_XLSX)
print("Loaded rows — Phoenix:", len(df_phoenix), "Fairbanks:", len(df_fairbanks))
print("Qserver first hour — Phoenix:", float(df_phoenix.loc[0, "Qserver_kW"]), "Fairbanks:", float(df_fairbanks.loc[0, "Qserver_kW"]))

# =============================================================================
# BLOCK 3 — PLANT MODEL (CRAH/WSE/CHILLER)
# Purpose: implement the equations you cite in §3.2 (hourly dispatch).
# =============================================================================
WATER_CP_KJ_PER_KG_K = 4.186
WATER_DENS_KG_PER_M3 = 1000.0

def k_cubic(x):
    """Vectorized cubic VFD scaling for fans/pumps (clamped 0..1)."""
    x = np.asarray(x, dtype=float)
    return np.clip(x, 0.0, 1.0) ** 3

def k_cubic_floor(x, power_floor=0.20):
    """Affinity with a non-zero parasitic floor if you want it; currently not used in final calc."""
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 0.0, 1.0)
    return power_floor + (1.0 - power_floor) * (x**3)

def hx_eps_ntu_Q(C_hot, C_cold, UA_kW_per_K, Th_in, Tc_in):
    """
    ε–NTU HX (both-mixed approximation), vectorized.
    Inputs: C_hot, C_cold in kW/K; UA in kW/K; temperatures in °C.
    Returns: (Q_kW, Th_out_C, Tc_out_C)
    """
    C_hot  = np.asarray(C_hot,  float)
    C_cold = np.asarray(C_cold, float)
    Th_in  = np.asarray(Th_in,  float)
    Tc_in  = np.asarray(Tc_in,  float)
    UA     = float(UA_kW_per_K)
    eps    = 1e-9

    C_min = np.minimum(C_hot, C_cold)
    C_max = np.maximum(C_hot, C_cold)
    Cr    = np.divide(C_min, C_max, out=np.zeros_like(C_min), where=(C_max > eps))
    NTU   = UA / np.maximum(C_min, eps)
    term  = np.exp(-NTU * (1.0 - Cr))
    denom = 1.0 - Cr * term
    eff   = np.divide(1.0 - term, denom, out=np.zeros_like(denom), where=(denom > eps))

    Qmax   = C_min * np.maximum(0.0, Th_in - Tc_in)
    Q      = eff * Qmax
    Th_out = np.where(C_hot  > eps, Th_in - Q / np.maximum(C_hot,  eps), Th_in)
    Tc_out = np.where(C_cold > eps, Tc_in + Q / np.maximum(C_cold, eps), Tc_in)
    return Q, Th_out, Tc_out

# Chiller COP polynomial surfaces (paper Eq. 4 & 5)
def cop_centrifugal(Tctw_C, PLR):
    b1, b2, b3, b4, b5, b6 = 25.47, -1.066, 6.335, 0.01188, 0.1263, -7.581
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return b1 + b2*T + b3*L + b4*T*T + b5*T*L + b6*L*L

# ✅ Keep only the correct magnetic COP (remove the buggy duplicate)
def cop_magnetic(Tctw_C, PLR):
    c1, c2, c3, c4, c5, c6 = 25.17, -0.7536, -1.081, 0.005736, 0.1851, -4.568
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return c1 + c2*T + c3*L + c4*T*T + c5*T*L + c6*L*L

@dataclass
class PlantParams:
    # Water-side setpoints and tower behavior
    Tcw_supply_set_C: float = 7.0
    dT_chilled_K: float     = 5.0
    dT_cooling_K: float     = 5.0
    tower_approach_K: float = 4.0
    tower_Tmin_C: float     = 2.0

    # Water-side economizer (plate HX)
    UA_wse_kW_per_K: float = 4000.0   # UA in kW/K

    # CRAH bank (capacities & fan power)
    crah_caps_kW: tuple = (72*102.0, 29*62.0, 12*20.5)
    crah_fans_kW: tuple = (72*5.66,  29*3.44, 12*1.40)

    # Pumps (rated flows & powers)
    chw_flow_m3_per_h_per_pump: float = 660.0
    chw_pumps_use: int                = 2
    chw_pump_power_kW_per_pump: float = 75.0

    ctw_flow_m3_per_h_per_pump: float = 900.0
    ctw_pumps_use: int                = 2
    ctw_pump_power_kW_per_pump: float = 55.0

    # Tower fans (rated)
    tower_fan_power_kW_per_ct: float = 37.5
    tower_cells_use: int             = 2

    # Chillers
    chiller_type: str               = "centrifugal"   # "centrifugal" or "magnetic"
    chiller_units_use: int          = 2
    chiller_cap_kW_per_unit: float  = 4058.0          # 4058 cen; 3900 mag

    COP_min: float = 2.0
    COP_max: float = 18.0

def plan_power_timeseries(df: pd.DataFrame, params: PlantParams) -> pd.DataFrame:
    """
    Hourly dispatch (8760 rows expected):
    Inputs per row: Qserver_kW, Twb_out_C   (and Tdb_out_C for reference)
    Outputs per row: mode label, Q splits, COP, component powers, P_sys_kW.
    """
    out = df.copy()

    # Total cooling load (Eq. 2): IT + ~10% facility overhead
    Qsum = out['Qserver_kW'].to_numpy(float) * 1.10

    # CRAH fan power (affinity law)
    crah_Q_cap     = sum(params.crah_caps_kW)
    crah_fan_rated = sum(params.crah_fans_kW)
    kfan_crah      = np.clip(Qsum / max(1.0, crah_Q_cap), 0, 1)
    P_crah         = k_cubic(kfan_crah) * crah_fan_rated

    # Required chilled-water flow to satisfy Qsum at ΔT_chilled
    Cpw          = WATER_CP_KJ_PER_KG_K
    m_chw_req    = Qsum / (Cpw * params.dT_chilled_K)     # kg/s
    chw_total    = params.chw_flow_m3_per_h_per_pump * params.chw_pumps_use
    m_chw_rated  = chw_total * WATER_DENS_KG_PER_M3 / 3600.0
    kflow_chw    = np.clip(m_chw_req / max(1e-9, m_chw_rated), 0, 1)
    C_hot        = np.minimum(m_chw_req, m_chw_rated) * Cpw  # kW/K
    Tcw_return_C = params.Tcw_supply_set_C + params.dT_chilled_K

    # Tower outlet (condenser water in): Twb_out + approach, floored
    Tctw_in = np.maximum(out['Twb_out_C'].to_numpy(float) + params.tower_approach_K,
                         params.tower_Tmin_C)

    # Cooling-water side capacity (rated pumps)
    ctw_total     = params.ctw_flow_m3_per_h_per_pump * params.ctw_pumps_use
    m_ctw_rated   = ctw_total * WATER_DENS_KG_PER_M3 / 3600.0
    C_cold_rated  = m_ctw_rated * Cpw  # kW/K

    # WSE ε–NTU transfer
    Q_wse, Th_out, Tc_out = hx_eps_ntu_Q(
        C_hot=C_hot, C_cold=C_cold_rated, UA_kW_per_K=params.UA_wse_kW_per_K,
        Th_in=Tcw_return_C, Tc_in=Tctw_in
    )
    # Cap to setpoint and load
    Q_to_set = np.maximum(0.0, (Tcw_return_C - params.Tcw_supply_set_C) * C_hot)
    Q_wse    = np.minimum(np.minimum(Q_wse, Q_to_set), Qsum)

    # Mode logic + unmet load Q_need
    Q_need = np.maximum(0.0, Qsum - Q_wse)
    mode   = np.where(Q_wse >= Qsum - 1e-6, "MODE1_FULL_FREE",
             np.where(Q_wse >  1e-6,       "MODE2_PARTIAL_FREE", "MODE3_CHILLER"))

    # Chiller portion after WSE (no ATES in this block)
    Q_ch = Q_need.copy()

    # Chiller COP from condenser in temperature and PLR
    cap_unit  = params.chiller_cap_kW_per_unit
    total_cap = cap_unit * params.chiller_units_use
    PLR_total = np.clip(Q_ch / max(1e-9, total_cap), 1e-6, 1.0)

    if params.chiller_type.lower().startswith('mag'):
        COP = cop_magnetic(Tctw_in, PLR_total)
    else:
        COP = cop_centrifugal(Tctw_in, PLR_total)
    COP = np.clip(COP, params.COP_min, params.COP_max)

    P_ch = np.where(Q_ch > 0.0, Q_ch / COP, 0.0)

    # Tower rejection and auxiliaries
    Q_rej         = Q_wse + np.where(Q_ch > 0.0, Q_ch * (COP - 1.0) / COP, 0.0)
    Q_tower_rated = C_cold_rated * params.dT_cooling_K
    ktower        = np.clip(Q_rej / max(Q_tower_rated, 1e-9), 0, 1)
    kflow_ctw     = ktower

    # ✅ MOVED BELOW (after kflow_chw, kflow_ctw, ktower exist) — fixes UnboundLocalError
    P_tower_fans  = k_cubic(ktower)     * (params.tower_fan_power_kW_per_ct * params.tower_cells_use)
    P_chw_pumps   = k_cubic(kflow_chw)  * (params.chw_pump_power_kW_per_pump * params.chw_pumps_use)
    P_ctw_pumps   = k_cubic(kflow_ctw)  * (params.ctw_pump_power_kW_per_pump * params.ctw_pumps_use)

    P_sys = P_ch + P_chw_pumps + P_ctw_pumps + P_tower_fans + P_crah

    out = out.assign(
        Qsum_kW=Qsum,
        Q_need_kW=Q_need,
        mode=mode,
        Q_wse_kW=Q_wse,
        Q_chiller_kW=Q_ch,
        COP=np.where(Q_ch > 0.0, COP, np.nan),
        P_chiller_kW=P_ch,
        P_crah_kW=P_crah,
        P_pumps_chw_kW=P_chw_pumps,
        P_pumps_ctw_kW=P_ctw_pumps,
        P_tower_fans_kW=P_tower_fans,
        P_sys_kW=P_sys,
        T_ctw_in_C=Tctw_in,
        T_cw_return_C=Tcw_return_C,
        T_cw_after_WSE_C=(Tcw_return_C - np.where(C_hot>0, Q_wse/np.maximum(C_hot,1e-9), 0.0))
    )
    return out

# =============================================================================
# BLOCK 4 — EXEC KPIs + PLAN COMPARISON
# Purpose: compute side-by-side hourly power and annual kWh/$ for Cen vs Mag.
# =============================================================================
def add_exec_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Qserver_kW']       = out['Qsum_kW'] / 1.10
    out['P_chiller_kW_est'] = np.where(out['Q_chiller_kW']>0, out['Q_chiller_kW']/out['COP'], 0.0)
    out['P_aux_kW']         = out['P_sys_kW'] - out['P_chiller_kW_est']
    out['Cooling_overhead_%'] = 100.0 * out['P_sys_kW'] / out['Qserver_kW']
    out['PUE_prime']        = 1.0 + out['P_sys_kW'] / out['Qserver_kW']
    return out

def compare_plans(df_hourly, p_mag, p_cen, elec_price: float, dt_hours: float = 1.0):
    res_mag = plan_power_timeseries(df_hourly, p_mag)
    res_cen = plan_power_timeseries(df_hourly, p_cen)

    view = pd.DataFrame({
        'mode': res_mag['mode'],
        'IT_kW': res_mag['Qsum_kW'] / 1.10,
        'Twb_out_C': df_hourly['Twb_out_C'],
        'P_sys_mag_kW': res_mag['P_sys_kW'],
        'P_sys_cen_kW': res_cen['P_sys_kW'],
    })
    view['Δ_kW (cen - mag)'] = view['P_sys_cen_kW'] - view['P_sys_mag_kW']

    kWh_mag   = float((res_mag['P_sys_kW'] * dt_hours).sum())
    kWh_cen   = float((res_cen['P_sys_kW'] * dt_hours).sum())
    delta_kWh = kWh_cen - kWh_mag
    delta_usd = delta_kWh * float(elec_price)
    pct_sav   = 100.0 * delta_kWh / kWh_cen if kWh_cen > 0 else np.nan

    totals = {
        "Magnetic total kWh":     kWh_mag,
        "Centrifugal total kWh":  kWh_cen,
        "ΔkWh (cen - mag)":       delta_kWh,
        "ΔUSD (cen - mag)":       delta_usd,
        "% savings vs centrifugal": pct_sav
    }
    return res_mag, res_cen, view, totals

# =============================================================================
# BLOCK 5 — RUN (8760 outputs) AND PRINT ANNUAL KPIs
# Purpose: produce exactly 8760-row hourly tables + compact annual summaries.
# =============================================================================
# Plant parameter bundles (only difference is chiller type/capacity)
p_mag = PlantParams(chiller_type="magnetic",    chiller_cap_kW_per_unit=3900.0)
p_cen = PlantParams(chiller_type="centrifugal", chiller_cap_kW_per_unit=4058.0)

# --- Phoenix
res_mag_phx, res_cen_phx, view_phx, totals_phx = compare_plans(
    df_phoenix, p_mag, p_cen, elec_price=SITES["Phoenix"]["energy_price_usd_per_kWh"], dt_hours=1.0
)
print("\n=== PHOENIX — Hourly (first 6 of 8760) ===")
print(view_phx.head(6))
print("\n=== PHOENIX — Annual totals ===")
for k, v in totals_phx.items():
    print(f"{k}: {v:,.2f}")

# --- Fairbanks
res_mag_fb, res_cen_fb, view_fb, totals_fb = compare_plans(
    df_fairbanks, p_mag, p_cen, elec_price=SITES["Fairbanks"]["energy_price_usd_per_kWh"], dt_hours=1.0
)
print("\n=== FAIRBANKS — Hourly (first 6 of 8760) ===")
print(view_fb.head(6))
print("\n=== FAIRBANKS — Annual totals ===")
for k, v in totals_fb.items():
    print(f"{k}: {v:,.2f}")


ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.